# Prepare CADD-SV v2 annotation bundle

One-time Terra / Workbench notebook to download the CADD-SV v2 annotation
bundle, package it as a `.tar.gz`, and upload it to the workspace bucket for
`AnnotateSvCallset.caddsv_annotations_tar`.

**Requirements**
- Persistent disk with **≥60 GB free** (uncompressed bundle is tens of GB)
- Network access to `kircherlab.bihealth.org`
- `gsutil` / workspace bucket write access

This notebook does **not** require Docker. It installs `caddsv` into a
**dedicated conda env under `caddsv_bundle_work/`** (not the Terra kernel)
and runs `caddsv get annotations`.

Do not `pip install caddsv` into the Workbench kernel: Terra's Click/Typer
mix crashes `caddsv --help`, and conda Python often cannot bootstrap `pip`
inside a stdlib `venv`.

In [ ]:
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path

BUNDLE_NAME = "caddsv-v2.0-annotations.tar.gz"
CADD_SV_VERSION = "2.0.2"
MIN_FREE_GB = 60

WORK = Path(os.environ.get("CADD_SV_WORK", Path.cwd() / "caddsv_bundle_work")).resolve()
CONDA_ENV = WORK / "caddsv-conda"
PYTHON = CONDA_ENV / "bin" / "python"
READY_MARKER = CONDA_ENV / ".caddsv_ready"
ANNOTATIONS_DIR = WORK / "annotations"
LOCAL_TAR = WORK / BUNDLE_NAME


def caddsv_cmd(*args: str) -> list[str]:
    """Invoke the CADD-SV Typer app without relying on bin/caddsv existing."""
    return [str(PYTHON), "-m", "caddsv.cli", *args]

WORKSPACE_BUCKET = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
GCS_DEST = os.environ.get(
    "CADD_SV_GCS_DEST",
    f"{WORKSPACE_BUCKET}/refs/caddsv/{BUNDLE_NAME}" if WORKSPACE_BUCKET else "",
)

print("WORK:", WORK)
print("PYTHON:", PYTHON)
print("CLI:", " ".join(caddsv_cmd("get", "annotations", "...")))
print("GCS_DEST:", GCS_DEST or "(set WORKSPACE_BUCKET or CADD_SV_GCS_DEST)")

In [ ]:
def free_gb(path: Path) -> float:
    path.mkdir(parents=True, exist_ok=True)
    usage = shutil.disk_usage(path)
    return usage.free / (1024 ** 3)


available = free_gb(WORK)
print(f"Free disk at {WORK}: {available:.1f} GB")
if available < MIN_FREE_GB:
    raise SystemExit(
        f"Need ≥{MIN_FREE_GB} GB free to download and package CADD-SV annotations; "
        f"have {available:.1f} GB. Attach a larger persistent disk and rerun."
    )

In [ ]:
def run(cmd: list[str], *, check: bool = True) -> subprocess.CompletedProcess[str]:
    print("+", " ".join(map(str, cmd)))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.stdout:
        print(proc.stdout, end="" if proc.stdout.endswith("\n") else "\n")
    if proc.stderr:
        print(proc.stderr, end="" if proc.stderr.endswith("\n") else "\n")
    if check and proc.returncode != 0:
        raise subprocess.CalledProcessError(
            proc.returncode, cmd, output=proc.stdout, stderr=proc.stderr
        )
    return proc


def env_is_ready() -> bool:
    if not PYTHON.is_file() or not READY_MARKER.is_file():
        return False
    probe = subprocess.run(
        [str(PYTHON), "-c", "import caddsv.cli"],
        capture_output=True,
        text=True,
    )
    return probe.returncode == 0


def ensure_caddsv_env() -> None:
    # Drop a broken stdlib venv from an earlier notebook version.
    legacy_venv = WORK / "venv"
    if legacy_venv.exists():
        print(f"Removing legacy venv at {legacy_venv}")
        shutil.rmtree(legacy_venv)

    if env_is_ready():
        print(f"Reusing conda env at {CONDA_ENV}")
        return

    if CONDA_ENV.exists():
        print(f"Removing incomplete env at {CONDA_ENV}")
        shutil.rmtree(CONDA_ENV)

    print(f"Creating conda env at {CONDA_ENV}")
    run(
        [
            "conda",
            "create",
            "-y",
            "-p",
            str(CONDA_ENV),
            "python=3.12",
            "pip",
        ]
    )
    run(
        [
            str(PYTHON),
            "-m",
            "pip",
            "install",
            "--upgrade",
            "pip",
            "wheel",
            f"caddsv=={CADD_SV_VERSION}",
            "typer>=0.16",
        ]
    )
    run(
        [
            str(PYTHON),
            "-c",
            (
                "import caddsv, caddsv.cli, click, typer; "
                "print('caddsv', getattr(caddsv, '__version__', 'ok')); "
                "print('typer', typer.__version__); "
                "print('click', click.__version__)"
            ),
        ]
    )
    READY_MARKER.write_text(f"{CADD_SV_VERSION}\n")


ensure_caddsv_env()
print("using", " ".join(caddsv_cmd()))

In [ ]:
if ANNOTATIONS_DIR.exists() and any(ANNOTATIONS_DIR.iterdir()):
    print(f"Reusing existing annotations at {ANNOTATIONS_DIR}")
else:
    print("Downloading CADD-SV v2 annotations (tens of GB; can take a while)...")
    run(
        caddsv_cmd(
            "get",
            "annotations",
            "--annotations-dir",
            str(ANNOTATIONS_DIR),
        )
    )

print("Annotation tree size:")
run(["du", "-sh", str(ANNOTATIONS_DIR)])

In [ ]:
if LOCAL_TAR.exists():
    print(f"Removing previous archive {LOCAL_TAR}")
    LOCAL_TAR.unlink()

print(f"Packaging {LOCAL_TAR} ...")
subprocess.check_call(
    ["tar", "-C", str(WORK), "-czf", str(LOCAL_TAR), "annotations"]
)
print("Archive size:")
subprocess.check_call(["ls", "-lh", str(LOCAL_TAR)])

In [ ]:
if not GCS_DEST:
    raise SystemExit(
        "Set WORKSPACE_BUCKET (Terra default) or CADD_SV_GCS_DEST before uploading."
    )

print(f"Uploading to {GCS_DEST}")
subprocess.check_call(["gsutil", "-m", "cp", str(LOCAL_TAR), GCS_DEST])
subprocess.check_call(["gsutil", "ls", "-lh", GCS_DEST])
print("\nUse this WDL input:")
print(f'  "AnnotateSvCallset.caddsv_annotations_tar": "{GCS_DEST}"')

## Optional cleanup

After a successful upload you can delete the local work directory to reclaim disk:

In [ ]:
# shutil.rmtree(WORK)
# print("Removed", WORK)